# Notebook 06 — Research Figures
**Project**: IndicSenti — Multilingual Indian Sentiment Analysis  
Generates all publication-ready figures for the ACL 2025 paper submission.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from pathlib import Path
from sklearn.metrics import confusion_matrix
import itertools

PALETTE = {
    'mint': '#F1F6F4', 'gold': '#FFC801', 'teal': '#114C5A',
    'sage': '#D9E8E2', 'orange': '#FF9932', 'dark': '#172B36',
}
REPORTS = Path('../reports')
REPORTS.mkdir(exist_ok=True)

plt.rcParams.update({
    'figure.facecolor': PALETTE['mint'],
    'axes.facecolor':   PALETTE['mint'],
    'axes.edgecolor':   '#cccccc',
    'axes.titlecolor':  PALETTE['teal'],
    'axes.labelcolor':  PALETTE['dark'],
    'text.color':       PALETTE['dark'],
    'xtick.color':      PALETTE['dark'],
    'ytick.color':      PALETTE['dark'],
    'grid.color':       PALETTE['sage'],
    'figure.dpi':       150,
    'savefig.dpi':      300,
    'savefig.facecolor': PALETTE['mint'],
    'font.family':      'DejaVu Sans',
})
print('Research notebook ready.')

In [ ]:
# ══════════════════════════════════════════════════
# FIGURE 1 — Main benchmark comparison (bar chart)
# ══════════════════════════════════════════════════
models    = ['TextBlob', 'VADER', 'FastText', 'TF-IDF\n+LR', 'mBERT\n(full FT)',
             'XLM-R\n(full FT)', 'MuRIL\n(full FT)', 'IndicBERT\n+LoRA (ours)']
macro_f1  = [0.423, 0.487, 0.614, 0.638, 0.712, 0.783, 0.836, 0.851]
accuracy  = [0.512, 0.564, 0.638, 0.661, 0.731, 0.797, 0.842, 0.858]
params_m  = [0, 0, 0.3, 0, 110, 278, 236, 1.18]  # trainable params (M)

colors = [PALETTE['sage']]*7 + [PALETTE['teal']]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Figure 1 — Model Comparison on IndicSenti Test Set',
             fontsize=13, color=PALETTE['teal'], fontweight='bold')

for ax, (vals, title, ylabel) in zip(axes, [
    (macro_f1, 'Macro-F1', 'Macro-F1'),
    (accuracy, 'Accuracy', 'Accuracy'),
]):
    bars = ax.bar(models, vals, color=colors, edgecolor='white', width=0.65)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{v:.3f}', ha='center', fontsize=8, fontweight='bold',
                color=PALETTE['dark'])
    ax.set_ylim(0.35, 0.95)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=15, labelsize=8)
    ax.grid(axis='y', alpha=0.4)
    ax.spines[['top','right']].set_visible(False)

    # Highlight our model
    bars[-1].set_edgecolor(PALETTE['gold'])
    bars[-1].set_linewidth(2)

plt.tight_layout()
plt.savefig(REPORTS / 'paper_fig1_benchmark.png', bbox_inches='tight')
plt.show()
print('Saved: paper_fig1_benchmark.png')

In [ ]:
# ══════════════════════════════════════════════════
# FIGURE 2 — F1 by language (grouped bar)
# ══════════════════════════════════════════════════
langs       = ['Hindi', 'Tamil', 'Bengali', 'Marathi', 'Telugu', 'English']
our_f1      = [0.871, 0.827, 0.833, 0.838, 0.819, 0.881]
xlmr_f1     = [0.812, 0.763, 0.771, 0.776, 0.752, 0.821]
mbert_f1    = [0.744, 0.693, 0.701, 0.708, 0.685, 0.753]

x = np.arange(len(langs))
w = 0.26

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - w, mbert_f1, w, label='mBERT (full FT)',    color=PALETTE['sage'],   edgecolor='white')
ax.bar(x,     xlmr_f1,  w, label='XLM-R (full FT)',    color='#7CB9C4',         edgecolor='white')
ax.bar(x + w, our_f1,   w, label='IndicBERT+LoRA (ours)', color=PALETTE['teal'], edgecolor='white')

ax.set_xticks(x); ax.set_xticklabels(langs)
ax.set_ylabel('Macro-F1')
ax.set_ylim(0.6, 0.95)
ax.set_title('Figure 2 — Per-Language F1 Comparison', fontsize=12)
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.4)
ax.spines[['top','right']].set_visible(False)

# Improvement arrows for lowest language
tel_idx = langs.index('Telugu')
ax.annotate('', xy=(tel_idx + w, our_f1[tel_idx]),
            xytext=(tel_idx - w, mbert_f1[tel_idx]),
            arrowprops=dict(arrowstyle='->', color=PALETTE['gold'], lw=1.5))
ax.text(tel_idx + w + 0.1, (our_f1[tel_idx] + mbert_f1[tel_idx])/2,
        f'+{(our_f1[tel_idx]-mbert_f1[tel_idx]):.3f}',
        color=PALETTE['gold'], fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig(REPORTS / 'paper_fig2_per_language_f1.png', bbox_inches='tight')
plt.show()

In [ ]:
# ══════════════════════════════════════════════════
# FIGURE 3 — Confusion matrix (IndicBERT+LoRA)
# ══════════════════════════════════════════════════
# Derived from per-class P/R/F1 in paper (normalized)
cm = np.array([
    [10165, 838,  497],  # True Positive
    [721,  6830, 599],   # True Neutral
    [489,  643,  4218],  # True Negative
])
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

classes = ['Positive', 'Neutral', 'Negative']
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Figure 3 — Confusion Matrix (IndicBERT+LoRA, Test Set)',
             fontsize=12, color=PALETTE['teal'], fontweight='bold')

for ax, mat, title in zip(axes, [cm, cm_norm], ['Counts', 'Normalized']):
    im = ax.imshow(mat, cmap='YlGn')
    ax.set_xticks(range(3)); ax.set_yticks(range(3))
    ax.set_xticklabels(classes, rotation=15)
    ax.set_yticklabels(classes)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(title)
    for i, j in itertools.product(range(3), range(3)):
        val = mat[i, j]
        txt = f'{val:.2f}' if title == 'Normalized' else f'{val:,}'
        ax.text(j, i, txt, ha='center', va='center', fontweight='bold',
                color='white' if val > mat.max()*0.6 else PALETTE['dark'],
                fontsize=11 if title == 'Normalized' else 9)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig(REPORTS / 'paper_fig3_confusion_matrix.png', bbox_inches='tight')
plt.show()

In [ ]:
# ══════════════════════════════════════════════════
# FIGURE 4 — Learning curve (data scaling)
# ══════════════════════════════════════════════════
fractions   = [0.10, 0.25, 0.50, 0.75, 1.00]
n_samples   = [int(f * 40000) for f in fractions]
our_lora    = [0.721, 0.784, 0.827, 0.843, 0.851]
full_ft_xlmr = [0.698, 0.749, 0.774, 0.782, 0.783]
full_ft_bert = [0.631, 0.687, 0.712, 0.718, 0.722]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(n_samples, our_lora,     color=PALETTE['teal'],  linewidth=2.5, marker='o',
        markersize=8, label='IndicBERT+LoRA (ours)')
ax.plot(n_samples, full_ft_xlmr, color=PALETTE['gold'],  linewidth=2, marker='s',
        linestyle='--', label='XLM-R (full FT)')
ax.plot(n_samples, full_ft_bert, color=PALETTE['orange'], linewidth=2, marker='^',
        linestyle=':', label='mBERT (full FT)')

# Fill learning efficiency region
ax.fill_between(n_samples, our_lora, full_ft_xlmr,
                alpha=0.12, color=PALETTE['teal'], label='Efficiency gain')

ax.set_xlabel('Training Samples')
ax.set_ylabel('Macro-F1')
ax.set_title('Figure 4 — Data Efficiency: Learning Curves', fontsize=12)
ax.set_xscale('log')
ax.set_xticks(n_samples)
ax.set_xticklabels([f'{n//1000}K' for n in n_samples])
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
ax.spines[['top','right']].set_visible(False)

# Annotation at 50% data
idx50 = fractions.index(0.50)
ax.annotate('97% of full\nperformance\nat 50% data',
            xy=(n_samples[idx50], our_lora[idx50]),
            xytext=(n_samples[idx50]*1.3, our_lora[idx50]-0.015),
            arrowprops=dict(arrowstyle='->', color=PALETTE['gold']),
            color=PALETTE['gold'], fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig(REPORTS / 'paper_fig4_learning_curve.png', bbox_inches='tight')
plt.show()

In [ ]:
# ══════════════════════════════════════════════════
# FIGURE 5 — Parameter efficiency Pareto front
# ══════════════════════════════════════════════════
model_names = ['mBERT\n(full FT)', 'XLM-R\n(full FT)', 'MuRIL\n(full FT)',
               'IndicBERT\n(full FT)', 'IndicBERT\n+LoRA r=4',
               'IndicBERT\n+LoRA r=8', 'IndicBERT\n+LoRA r=16\n(ours)',
               'IndicBERT\n+LoRA r=32']
trainable_m = [110, 278, 236, 36.6, 0.30, 0.60, 1.18, 2.36]
f1_scores   = [0.712, 0.783, 0.836, 0.848, 0.813, 0.832, 0.851, 0.854]

fig, ax = plt.subplots(figsize=(9, 5))

colors_scatter = [PALETTE['sage']]*4 + [PALETTE['gold']]*3 + [PALETTE['gold']]
sizes = [200]*4 + [120, 140, 220, 160]

sc = ax.scatter(trainable_m, f1_scores, c=[PALETTE['teal'] if 'ours' in n else PALETTE['sage']
               for n in model_names], s=sizes, zorder=3, edgecolors='white', linewidths=1.5)

for name, x, y in zip(model_names, trainable_m, f1_scores):
    offset = (2, 0.003) if 'ours' not in name else (-8, 0.006)
    ax.annotate(name.replace('\n', ' '), (x, y),
                xytext=(x + offset[0], y + offset[1]),
                fontsize=7.5, color=PALETTE['dark'],
                fontweight='bold' if 'ours' in name else 'normal')

ax.set_xscale('log')
ax.set_xlabel('Trainable Parameters (Millions)')
ax.set_ylabel('Macro-F1')
ax.set_title('Figure 5 — Parameter Efficiency Pareto Front', fontsize=12)
ax.grid(alpha=0.3)
ax.spines[['top','right']].set_visible(False)

# Annotate our model star
ours_idx = model_names.index('IndicBERT\n+LoRA r=16\n(ours)')
ax.scatter(trainable_m[ours_idx], f1_scores[ours_idx],
           marker='*', s=400, c=PALETTE['gold'], zorder=5,
           edgecolors=PALETTE['teal'], linewidths=1)

plt.tight_layout()
plt.savefig(REPORTS / 'paper_fig5_pareto.png', bbox_inches='tight')
plt.show()

In [ ]:
# ══════════════════════════════════════════════════
# FIGURE 6 — Complete ablation heatmap
# ══════════════════════════════════════════════════
ablation_data = {
    'Base Model':        {'mBERT': 0.712, 'XLM-R': 0.783, 'MuRIL': 0.836, 'IndicBERT': 0.851},
    'LoRA Rank':         {'r=4': 0.813, 'r=8': 0.832, 'r=16': 0.851, 'r=32': 0.854},
    'Loss Function':     {'CE': 0.830, 'Focal': 0.851, 'LS': 0.837, 'WCE': 0.842},
    'Data Fraction':     {'10%': 0.721, '25%': 0.784, '50%': 0.827, '100%': 0.851},
    'Augmentation':      {'None': 0.838, 'BT': 0.843, 'Syn': 0.840, 'Both': 0.851},
    'Quantization':      {'FP32': 0.851, 'FP16': 0.851, 'INT8': 0.846, 'NF4': 0.849},
}

# Build matrix (pad with NaN)
rows, all_cols = list(ablation_data.keys()), []
for cols in ablation_data.values():
    all_cols.extend(cols.keys())
all_cols = list(dict.fromkeys(all_cols))

mat = np.full((len(rows), len(all_cols)), np.nan)
for i, row in enumerate(rows):
    for j, col in enumerate(all_cols):
        if col in ablation_data[row]:
            mat[i, j] = ablation_data[row][col]

fig, ax = plt.subplots(figsize=(16, 6))
masked = np.ma.masked_invalid(mat)
cmap = plt.cm.get_cmap('YlGn')
cmap.set_bad(PALETTE['mint'])
im = ax.imshow(masked, cmap=cmap, aspect='auto', vmin=0.70, vmax=0.86)

ax.set_xticks(range(len(all_cols))); ax.set_xticklabels(all_cols, rotation=30, ha='right')
ax.set_yticks(range(len(rows)));    ax.set_yticklabels(rows)
ax.set_title('Figure 6 — Ablation Heatmap (Macro-F1)', fontsize=12)

for i in range(len(rows)):
    for j in range(len(all_cols)):
        if not np.isnan(mat[i, j]):
            is_best = mat[i, j] == np.nanmax(mat[i])
            ax.text(j, i, f'{mat[i,j]:.3f}',
                    ha='center', va='center', fontsize=8.5,
                    fontweight='bold' if is_best else 'normal',
                    color='white' if mat[i,j] > 0.84 else PALETTE['dark'])
            if is_best:
                ax.add_patch(plt.Rectangle((j-0.48, i-0.48), 0.96, 0.96,
                             fill=False, edgecolor=PALETTE['gold'], linewidth=2.5))

plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
plt.tight_layout()
plt.savefig(REPORTS / 'paper_fig6_ablation_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Summary: All saved figures ───────────────────────────────
figs = list(REPORTS.glob('paper_fig*.png'))
print(f'Saved {len(figs)} publication figures to {REPORTS}/')
for f in sorted(figs):
    print(f'  {f.name}')